#### Quick diagnostic script â€” run interactively to debug training issues.

In [10]:
import torch
import torch.nn.functional as F
import sys, os

PROJECT_ROOT = r"C:\Users\Devesh\OneDrive\Desktop\project"
sys.path.insert(0, PROJECT_ROOT)

from models.snn_autoencoder import SNNAutoencoder
from preprocessing.event_dataset_torch import EventSpikeDataset
from torch.utils.data import DataLoader

#### Load a small batch

In [11]:
dataset = EventSpikeDataset(
    event_dir=os.path.join(PROJECT_ROOT, "events", "train"),
    target_size=(128, 128),
    num_steps=25,
    encoding="rate"
)

loader = DataLoader(dataset, batch_size=4, shuffle=True)
batch = next(iter(loader))

print(f"Batch shape: {batch.shape}")
print(f"Batch min/max: {batch.min():.4f} / {batch.max():.4f}")
print(f"Batch mean spike rate: {batch.mean():.4f}")
print(f"Non-zero fraction: {(batch > 0).float().mean():.4f}")

EventSpikeDataset initialized:
  Files: 10
  Samples: 2040
  Target size: (128, 128)
  Timesteps: 25
  Encoding: rate
Batch shape: torch.Size([4, 2, 128, 128, 25])
Batch min/max: 0.0000 / 1.0000
Batch mean spike rate: 0.0017
Non-zero fraction: 0.0017


#### Forward pass

In [12]:
model = SNNAutoencoder(in_channels=2, num_steps=25)
spike_count_out, mem_final, spike_record = model(batch)

print(f"\nOutput spike count shape: {spike_count_out.shape}")
print(f"Output spike count min/max: {spike_count_out.min():.4f} / {spike_count_out.max():.4f}")
print(f"Spike record shape: {spike_record.shape}")
print(f"Output spike rate: {spike_record.mean():.4f}")


Output spike count shape: torch.Size([4, 2, 128, 128])
Output spike count min/max: 0.0000 / 25.0000
Spike record shape: torch.Size([25, 4, 2, 128, 128])
Output spike rate: 0.1965


#### Check if output has any activity

In [13]:
NUM_STEPS   = 25
TARGET_RATE = 0.05
RATE_WEIGHT = 1.0

if spike_record.sum() == 0:
    print("WARNING: No output spikes! Network is dead.")
    print("  Try beta=0.85 or beta=0.9 instead of 0.95")

# Normalise counts to rates (matches train_snn.py snn_loss)
spike_count_in = batch.sum(dim=-1)          # (B, C, H, W)
rate_out = spike_count_out / NUM_STEPS
rate_in  = spike_count_in  / NUM_STEPS
mse_loss  = F.mse_loss(rate_out, rate_in)

# Rate homeostasis term
actual_rate = spike_record.mean()
rate_loss   = RATE_WEIGHT * (actual_rate - TARGET_RATE) ** 2
total_loss  = mse_loss + rate_loss

print(f"Input  spike rate:  {rate_in.mean():.4f}")
print(f"Output spike rate:  {rate_out.mean():.4f}")
print(f"Actual firing rate: {actual_rate.item():.4f}  (target {TARGET_RATE})")
print(f"MSE Loss:           {mse_loss.item():.6f}")
print(f"Rate Loss:          {rate_loss.item():.6f}")
print(f"Total Loss:         {total_loss.item():.6f}")

Input  spike rate:  0.0017
Output spike rate:  0.1965
Actual firing rate: 0.1965  (target 0.05)
MSE Loss:           0.108040
Rate Loss:          0.021453
Total Loss:         0.129493


#### Gradient check

In [14]:
total_loss.backward()
total_grad = 0
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        total_grad += grad_norm
        if grad_norm == 0:
            print(f"  Zero gradient: {name}")

print(f"\nTotal gradient norm: {total_grad:.6f}")
if total_grad == 0:
    print("ALL gradients are zero — surrogate gradient not working")
    print("  Check snntorch version. Try: pip install snntorch --upgrade")


Total gradient norm: 105.363599
